In [15]:
pd.set_option('display.max_rows', 500)
pd.set_option('display.max_columns', 500)


In [4]:
# from pathlib import Path
# import pandas as pd

project_root = Path.cwd().parents[1]

bronze_path = project_root / "data" / "bronze" / "bronze_burvot_t1_rhone_69.csv"
silver_path = project_root / "data" / "silver" / "silver_burvot_t1_rhone_69.csv"

# Read bronze
df_bronze = pd.read_csv(bronze_path, sep=";", dtype=str, encoding="utf-8")

# Transform -> silver dataframe
df_silver = df_bronze.copy()

# ... your cleaning / renaming / casting on df_silver ...

# Save silver
df_silver.to_csv(silver_path, sep=";", index=False, encoding="utf-8")
print(f"Wrote silver: {silver_path}")


Wrote silver: /Users/zainfrayha/Documents/EPSI/MSPR/electio-analytics-poc/data/silver/silver_burvot_t1_rhone_69.csv


In [5]:
df_silver.dtypes

Code du département              str
Libellé du département           str
Code de la circonscription       str
Libellé de la circonscription    str
Code de la commune               str
                                ... 
% Voix/Ins.11                    str
% Voix/Exp.11                    str
extraction_source_url            str
ingestion_timestamp              str
source_file_name                 str
Length: 108, dtype: object

In [6]:
# Keep code columns as strings to preserve leading zeros
base_dtypes = {
    "Code du département": "string",
    "Libellé du département": "string",
    "Code de la circonscription": "string",
    "Libellé de la circonscription": "string",
    "Code de la commune": "string",
    "Libellé de la commune": "string",
    "Code du b.vote": "Float64",
    "Inscrits": "Int64",
    "Abstentions": "Int64",
    "% Abs/Ins": "Float64",
    "Votants": "Int64",
    "% Vot/Ins": "Float64",
    "Blancs": "Int64",
    "% Blancs/Ins": "Float64",
    "% Blancs/Vot": "Float64",
    "Nuls": "Int64",
    "% Nuls/Ins": "Float64",
    "% Nuls/Vot": "Float64",
    "Exprimés": "Int64",
    "% Exp/Ins": "Float64",
    "% Exp/Vot": "Float64",
}

# Candidate columns repeat with suffixes: '', '.1', '.2', ..., '.11'
candidate_dtypes = {}
for suffix in [""] + [f".{i}" for i in range(1, 12)]:
    candidate_dtypes.update({
        f"N°Panneau{suffix}": "Int64",
        f"Sexe{suffix}": "string",
        f"Nom{suffix}": "string",
        f"Prénom{suffix}": "string",
        f"Voix{suffix}": "Int64",
        f"% Voix/Ins{suffix}": "Float64",
        f"% Voix/Exp{suffix}": "Float64",
    })

meta_dtypes = {
    "extraction_source_url": "string",
    "ingestion_timestamp": "string",
    "source_file_name": "string",
}

dtype_map = {**base_dtypes, **candidate_dtypes, **meta_dtypes}

for col, dtype in dtype_map.items():
    if col not in df_silver.columns:
        continue

    if dtype in {"Int64", "Float64"}:
        df_silver[col] = pd.to_numeric(
            df_silver[col].astype("string").str.replace(",", ".", regex=False),
            errors="coerce",
        ).astype(dtype)
    else:
        df_silver[col] = df_silver[col].astype(dtype)


In [9]:
df_silver.dtypes


Code du département               string
Libellé du département            string
Code de la circonscription        string
Libellé de la circonscription     string
Code de la commune                string
Libellé de la commune             string
Code du b.vote                   Float64
Inscrits                           Int64
Abstentions                        Int64
% Abs/Ins                        Float64
Votants                            Int64
% Vot/Ins                        Float64
Blancs                             Int64
% Blancs/Ins                     Float64
% Blancs/Vot                     Float64
Nuls                               Int64
% Nuls/Ins                       Float64
% Nuls/Vot                       Float64
Exprimés                           Int64
% Exp/Ins                        Float64
% Exp/Vot                        Float64
N°Panneau                          Int64
Sexe                              string
Nom                               string
Prénom          

In [21]:
df_silver.head(10)
print(df_silver['Code de la commune'])

0       001
1       002
2       003
3       003
4       004
       ... 
1312    298
1313    298
1314    299
1315    299
1316    299
Name: Code de la commune, Length: 1317, dtype: string


In [ ]:
print(df_silver.columns.tolist())